# ScoreCompose-AI — Colab T4 학습 노트북

**런타임 → 런타임 유형 변경 → T4 GPU** 로 바꾼 뒤 위에서부터 차례로 실행하세요.

8 epoch 기준 T4에서 약 4–6시간 소요됩니다 (MAESTRO 기준; LMD-clean이면 더 짧음). 중간에 런타임이 끊겨도 마지막 체크포인트(`weights.pt.last`)에서 자동 재개됩니다.

**필요한 입력:** 없음. **출력:** Google Drive의 `MyDrive/score_compose_ai/weights.pt` + 샘플 MIDI/MusicXML.

## 1. GPU 확인

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

## 2. Google Drive 마운트 (체크포인트 영속화)

런타임이 끊기면 `/content`는 사라지지만 Drive에 저장한 가중치는 살아남습니다. **건너뛰려면 다음 셀을 실행하지 마세요** — 그러면 `/content/score_compose_ai/`에만 저장됩니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
OUT_ROOT = '/content/drive/MyDrive/score_compose_ai'
os.makedirs(OUT_ROOT, exist_ok=True)
print('output root:', OUT_ROOT)

## 3. 코드 가져오기 (GitHub clone)

기본 경로입니다. 이미 클론돼 있으면 건너뜁니다. 본인 fork를 쓰고 싶다면 아래 URL만 바꾸시면 됩니다.

In [ ]:
import os
REPO_URL = 'https://github.com/rosyrosys/score_compose_ai.git'
REPO_DIR = '/content/score_compose_ai'

if not os.path.isdir(REPO_DIR):
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    print('repo already present at', REPO_DIR)
    !cd "$REPO_DIR" && git pull --ff-only

%cd $REPO_DIR
!ls

## 4. 의존성 설치

Colab엔 이미 torch가 깔려 있으니 부족한 것만 설치합니다.

In [ ]:
!pip install -q music21 pretty_midi mido tqdm fastapi 'uvicorn[standard]' pydantic websockets soundfile pyfluidsynth
print('deps installed')

## 5. 데이터셋 다운로드

기본값 **MAESTRO** (피아노, ~80MB, 안정적). LMD-clean을 쓰려면 `--source lmd`로.

In [ ]:
!python scripts/prepare_lakh_subset.py --source maestro --out data/midi --max_files 5000
import os; print('files:', len(os.listdir('data/midi')))

## 6. 학습 — 8 epoch

`OUT_ROOT`이 정의돼 있으면 Drive에, 아니면 `/content`에 저장합니다.
체크포인트는 500 step마다 `weights.pt.last`에 갱신, epoch 끝마다 `weights.pt`에 저장.

In [ ]:
import os
OUT = os.path.join(globals().get('OUT_ROOT', '/content/score_compose_ai'), 'weights.pt')
RESUME = OUT + '.last' if os.path.exists(OUT + '.last') else ''
print('out =', OUT)
print('resume =', RESUME or '(none)')

!python -m src.train \
    --midi_dir data/midi \
    --out "$OUT" \
    --epochs 8 \
    --batch_size 16 \
    --seq_len 1024 \
    --ckpt_every 500 \
    --resume "$RESUME"

## 7. 샘플 생성으로 동작 확인

`sample.mid`, `sample.musicxml`을 만들어 다운로드합니다.

### 6.5 (선택) Smoke test — 1 epoch / 100 파일로 파이프라인만 빠르게 검증

T4에서 ~5–10분. 음악적 품질은 형편없지만 학습→체크포인트→샘플 흐름이 도는지 확인할 때 쓰세요. **본 학습(셀 6)을 이미 돌렸다면 이 셀은 건너뛰세요** — 약한 가중치로 덮어씁니다.

In [ ]:
import os, shutil

# Make a tiny subset (first 100 .mid files).
os.makedirs('data/midi_smoke', exist_ok=True)
all_files = sorted(f for f in os.listdir('data/midi') if f.endswith(('.mid', '.midi')))
for f in all_files[:100]:
    src = os.path.join('data/midi', f)
    dst = os.path.join('data/midi_smoke', f)
    if not os.path.exists(dst):
        shutil.copy2(src, dst)
print('smoke files:', len(os.listdir('data/midi_smoke')))

OUT = os.path.join(globals().get('OUT_ROOT', '/content/score_compose_ai'),
                   'weights_smoke.pt')
print('smoke weights ->', OUT)

!python -m src.train \
    --midi_dir data/midi_smoke \
    --out "$OUT" \
    --epochs 1 \
    --batch_size 8 \
    --seq_len 512 \
    --ckpt_every 200

In [ ]:
import os, subprocess

# Sanity checks before sampling.
assert 'OUT' in dir(), "Run cell 6 first so 'OUT' is defined."
print('weights path :', OUT)
print('weights exist:', os.path.exists(OUT))
if not os.path.exists(OUT):
    raise FileNotFoundError(
        f"No weights at {OUT}. Either training did not finish, "
        "or you skipped the training cell. Run cell 6 first."
    )

cmd = [
    'python', 'scripts/sample_after_training.py',
    '--weights', OUT,
    '--out_midi', 'sample.mid',
    '--out_xml',  'sample.musicxml',
    '--n_tokens', '600',
    '--temperature', '1.0',
    '--top_p', '0.92',
    '--tempo', '110',
]
print('running:', ' '.join(cmd))
r = subprocess.run(cmd)
if r.returncode != 0:
    raise RuntimeError(f"sample_after_training.py failed (exit {r.returncode}). "
                       "See traceback above.")

for f in ('sample.mid', 'sample.musicxml'):
    assert os.path.exists(f), f"{f} was not created"

from google.colab import files
files.download('sample.mid')
files.download('sample.musicxml')

## 8. (선택) 편집 latency 벤치마크

학습된 가중치로 표 1 (논문)의 수치를 재현해 보고 싶을 때.

In [ ]:
!python scripts/benchmark_edits.py

## 9. (선택) 서버를 Colab에서 띄우고 ngrok으로 외부에서 편집

런타임에서 직접 OSMD 편집기를 쓰고 싶을 때만. ngrok 토큰이 필요합니다.

In [ ]:
# !pip install -q pyngrok
# from pyngrok import ngrok
# ngrok.set_auth_token('YOUR_NGROK_TOKEN')
# import os; os.environ['SCORECOMPOSE_WEIGHTS'] = OUT
# import threading, uvicorn
# from src.server import app
# threading.Thread(target=lambda: uvicorn.run(app, host='0.0.0.0', port=8000), daemon=True).start()
# print('public URL:', ngrok.connect(8000).public_url)